In [ ]:
upstream = None
product = None
mcleish24_tables2 = None

# Experimental data: Libraries and hosts

In [46]:
import pandas as pd
import requests
import json
import re
import random
import datetime

In [53]:
def timestamp_code():
    """
    This function generates a time stamp that goes
    to microsecond level, and it adds a random code
    just in case
    """
    now = datetime.datetime.now()
    timestamp = now.strftime("%Y%m%d%H%M%S%f")
    random.seed(timestamp[-4:])
    random_code = "{:03d}".format(random.randint(0, 100))
    return timestamp +  random_code

print(timestamp_code())
print(timestamp_code())
for i in range(1000):
    if timestamp_code() == timestamp_code():
        raise RuntimeError("the function did not work")

20251006110317675943068
20251006110317676105082


## Loading data

In [ ]:
libraries = pd.read_csv(mcleish24_tables2, sep=';')
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


## Assign taxonomic categories

The following function should enable us to obtain the taxonomy codes by querying against the NCBI. 

In [14]:
def search_taxonomy(host_name):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2"
    url = f'{url}/taxonomy/taxon/{host_name}'.replace(' ', '%20')

    r = requests.get(url)
    try: 
        r.raise_for_status()
    except requests.HTTPError:
        return pd.NA
    try:
        u = str(r.json()['taxonomy_nodes'][0]['taxonomy']['tax_id'])
    except KeyError:
        return pd.NA
    except IndexError:
        return pd.NA
    return u

search_taxonomy("Retama sphaerocarpa") # 49838

'49838'

Now a bit of simple editing to ease the search. We are discarding variety names, and we are replacing the sp symbols by sp. using regex (otherwise, some species with internals "sp" might get a "sp.")

In [13]:
hosts = libraries.drop_duplicates(subset=['Host_taxon'])[['Host_taxon']]
hosts['Host_curated'] = hosts['Host_taxon'].apply(lambda x: re.sub("sp$", "sp.", x))
hosts['Host_curated'] = hosts['Host_curated'].apply(lambda x: " ".join(x.split(" ")[:2]))
hosts

,Host_taxon,Host_curated
0,Amaranthus sp,Amaranthus sp.
1,Convolvulus arvensis,Convolvulus arvensis
2,Cucumis melo,Cucumis melo
4,Cyperus longus,Cyperus longus
5,Artemisia herba alba,Artemisia herba
...,...,...
309,Lithodora fruticosa,Lithodora fruticosa
310,Origanum vulgare,Origanum vulgare
311,Rhamnus lycioides,Rhamnus lycioides
315,Astragalus sesameus,Astragalus sesameus


We convert the table into a deduplicated list of organisms names, and we search all of them against the NCBI. 

In [55]:

species_taxid_map = []
for item in hosts['Host_curated'].unique().tolist():
    species_taxid_map.append(
        dict(species_name=item, taxid=search_taxonomy(item), code=timestamp_code())
    )
    print(species_taxid_map[-1])
hosts = pd.merge(hosts, pd.DataFrame.from_records(species_taxid_map), left_on='Host_curated', right_on='species_name')
hosts = hosts[['Host_taxon', 'taxid', 'code']]
hosts

{'species_name': 'Amaranthus sp.', 'taxid': '3076757', 'code': '20251006110439666773010'}
{'species_name': 'Convolvulus arvensis', 'taxid': '4123', 'code': '20251006110440201955043'}
{'species_name': 'Cucumis melo', 'taxid': '3656', 'code': '20251006110440754997052'}
{'species_name': 'Cyperus longus', 'taxid': '76433', 'code': '20251006110441254052093'}
{'species_name': 'Artemisia herba', 'taxid': <NA>, 'code': '20251006110441672270075'}
{'species_name': 'Teucrium pseudochamaepitys', 'taxid': '1209883', 'code': '20251006110442177792043'}
{'species_name': 'Jasminum fruticans', 'taxid': <NA>, 'code': '20251006110442545729047'}
{'species_name': 'Quercus coccifera', 'taxid': '58335', 'code': '20251006110443041873018'}
{'species_name': 'Quercus ilex', 'taxid': '58334', 'code': '20251006110443582725004'}
{'species_name': 'Portulaca oleraceae', 'taxid': <NA>, 'code': '20251006110443948438099'}
{'species_name': 'Chenopodium album', 'taxid': '3559', 'code': '20251006110444445883041'}
{'species_

,Host_taxon,taxid,code
0,Amaranthus sp,3076757,20251006110439666773010
1,Convolvulus arvensis,4123,20251006110440201955043
2,Cucumis melo,3656,20251006110440754997052
3,Cyperus longus,76433,20251006110441254052093
4,Artemisia herba alba,<NA>,20251006110441672270075
...,...,...,...
114,Lithodora fruticosa,475890,20251006110540840025049
115,Origanum vulgare,39352,20251006110541477751038
116,Rhamnus lycioides,72169,20251006110541985242010
117,Astragalus sesameus,2014722,20251006110542533764051


In [56]:
hosts_library = pd.merge(hosts, libraries, on=['Host_taxon']).drop_duplicates(['code', 'Library_code'], keep='first')[['code', 'Library_code']].to_dict(orient='records')
hosts_library

[{'code': '20251006110439666773010', 'Library_code': 'PV001'},
 {'code': '20251006110439666773010', 'Library_code': 'PV033'},
 {'code': '20251006110439666773010', 'Library_code': 'PV175'},
 {'code': '20251006110439666773010', 'Library_code': 'PV196'},
 {'code': '20251006110439666773010', 'Library_code': 'PV212'},
 {'code': '20251006110440201955043', 'Library_code': 'PV002'},
 {'code': '20251006110440201955043', 'Library_code': 'PV031'},
 {'code': '20251006110440201955043', 'Library_code': 'PV079'},
 {'code': '20251006110440201955043', 'Library_code': 'PV097'},
 {'code': '20251006110440201955043', 'Library_code': 'PV132'},
 {'code': '20251006110440201955043', 'Library_code': 'PV142'},
 {'code': '20251006110440201955043', 'Library_code': 'PV154'},
 {'code': '20251006110440201955043', 'Library_code': 'PV161'},
 {'code': '20251006110440201955043', 'Library_code': 'PV188'},
 {'code': '20251006110440201955043', 'Library_code': 'PV201'},
 {'code': '20251006110440201955043', 'Library_code': 'P

In [ ]:
host_library = dict(
    host=hosts.to_dict(orient='records'),
    host_library=hosts_library
)
with open(product['data'], "w") as f:
    json.dump(host_library, f, indent=4, sort_keys=True)